# PSF diagnostics

Characterise the PSFs ShearNet trains against, and (optionally) measure a trained
model's **PSF leakage** — the spurious shear it inherits from PSF ellipticity and
size.

**Part 1 — PSF characterisation** (no model needed): simulate a PSF-carrying
dataset, view the PSF stamps, and look at the distribution of PSF ellipticity
(e1, e2) and size (T).

**Part 2 — PSF leakage** (optional): for a trained model, correlate predicted
shear with PSF properties, DES-Y3 style. A leakage-free estimator has slopes
α ≈ β ≈ 0.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from shearnet.core.dataset import generate_dataset, split_combined_images
from shearnet.plotting.psf import (
    extract_psf_properties_from_obs,
    plot_psf_systematics_from_eval,
    calculate_psf_leakage_coefficients,
)

## 1 · Simulate a PSF-carrying dataset

`return_psf=True` stacks the PSF stamp as a second image channel; `return_obs=True`
also returns the ngmix observations we measure PSF moments from. To get PSF
*variation* (needed for a leakage measurement) use `apply_psf_shear=True` with the
ideal PSF, or switch to the empirical `exp="superbit"` PSF.

In [ ]:
N = 1000
PSF_FWHM = 0.25
EXP = "ideal"            # "ideal" (analytic Gaussian) or "superbit" (empirical PSFEx)
APPLY_PSF_SHEAR = True   # vary the ideal PSF ellipticity so leakage is measurable
SEED = 7

images, labels, obs = generate_dataset(
    N, PSF_FWHM,
    exp=EXP, seed=SEED,
    return_psf=True, return_obs=True,
    apply_psf_shear=APPLY_PSF_SHEAR,
    output_keys=("g1", "g2"),
)
gal_images, psf_images = split_combined_images(images, has_psf=True)
print("galaxy:", gal_images.shape, "  psf:", psf_images.shape)

In [ ]:
# A few PSF stamps (log-stretched).
n_show = 6
sub = psf_images[:n_show]
vmax = float(sub.max())
vmin = max(float(sub[sub > 0].min()), vmax * 1e-4)
fig, axes = plt.subplots(1, n_show, figsize=(2.2 * n_show, 2.4))
for k in range(n_show):
    axes[k].imshow(psf_images[k], cmap="gray", norm=LogNorm(vmin=vmin, vmax=vmax))
    axes[k].set_title(f"PSF {k}", fontsize=9)
    axes[k].axis("off")
plt.tight_layout()
plt.show()

## 2 · PSF property distributions

`extract_psf_properties_from_obs` measures adaptive-moment ellipticity `(e1, e2)`
and size `T` for each PSF.

In [ ]:
psf_props = extract_psf_properties_from_obs(obs)
finite = (np.isfinite(psf_props["e1"]) & np.isfinite(psf_props["e2"])
          & np.isfinite(psf_props["T"]))
e1, e2, T = psf_props["e1"][finite], psf_props["e2"][finite], psf_props["T"][finite]
print(f"{finite.sum()} / {finite.size} PSFs with valid moments")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (arr, name) in zip(axes, [(e1, "PSF e1"), (e2, "PSF e2"), (T, "PSF T")]):
    ax.hist(arr, bins=40, alpha=0.8)
    ax.set_title(f"{name}   mean={arr.mean():.3e}, std={arr.std():.3e}")
    ax.set_xlabel(name)
    ax.set_ylabel("count")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(e1, e2, s=8, alpha=0.4)
plt.axhline(0, color="k", lw=0.5)
plt.axvline(0, color="k", lw=0.5)
plt.xlabel("PSF e1")
plt.ylabel("PSF e2")
plt.title("PSF ellipticity distribution")
plt.gca().set_aspect("equal", adjustable="box")
plt.grid(alpha=0.3)
plt.show()

## 3 · PSF leakage for a trained model (optional)

Point `MODEL_NAME` at a model trained on the **same** `exp` / `psf_fwhm` used
above. We predict shear on this PSF-carrying set and correlate predictions with
PSF properties. If no checkpoint is found the section is skipped.

In [ ]:
MODEL_NAME = "quickstart_demo"

DATA_PATH = os.getenv("SHEARNET_DATA_PATH", os.path.abspath("."))
SAVE_PATH = os.path.join(DATA_PATH, "model_checkpoint")
PLOT_PATH = os.path.join(DATA_PATH, "plots")
have_model = (os.path.isdir(SAVE_PATH)
              and any(d.startswith(MODEL_NAME) for d in os.listdir(SAVE_PATH)))
print("model found:", have_model)

In [ ]:
if have_model:
    from shearnet.config.config_handler import Config
    from shearnet.cli.evaluate import load_model, run_shearnet

    cfg_path = os.path.join(PLOT_PATH, MODEL_NAME, "training_config.yaml")
    config = Config(cfg_path) if os.path.exists(cfg_path) else Config()
    process_psf = config.get("model.process_psf", False)

    cfg = {
        "model_name": MODEL_NAME,
        "seed": SEED,
        "process_psf": process_psf,
        "nn": config.get("model.type", "cnn"),
        "galaxy_type": config.get("model.galaxy.type", "research_backed"),
        "psf_type": config.get("model.psf.type", "forklens_psf"),
        "fusion": config.get("model.fusion", "concat"),
        "output_keys": tuple(config.get("model.output_keys", ("g1", "g2"))),
        "gap": config.get("model.gap", False),
        "save_path": SAVE_PATH,
        "plot_path": PLOT_PATH,
    }
    psf_in = psf_images if process_psf else None
    state = load_model(cfg, gal_images, psf_in)
    preds, _, _, _ = run_shearnet(state, gal_images, psf_in, labels, cfg)
    preds = np.asarray(preds)

    # DES-Y3-style systematics panels (predicted shear vs PSF e1/e2/T).
    plot_psf_systematics_from_eval(obs, preds, n_bins=15)
    plt.show()

    # Leakage coefficients from the shear residual (pred - true).
    props_f = {k: psf_props[k][finite] for k in ("e1", "e2", "T")}
    coeffs = calculate_psf_leakage_coefficients(
        preds[finite, :2], props_f, true_shears=labels[finite, :2]
    )
    print("\nPSF leakage coefficients (residual vs PSF property):")
    for name, v in coeffs.items():
        if v is not None:
            print(f"  {name}: {v['value']:+.3e} ± {v['error']:.1e}")
else:
    print(f"No checkpoint for '{MODEL_NAME}'. Train one (see 01_quickstart.ipynb) "
          f"on exp='{EXP}', psf_fwhm={PSF_FWHM} to run this section.")